# Атака на оракул по младшему биту RSA 
Представьте, что у Вас есть побочный канал на сервере, принимающем шифротексты RSA, который позволяет Вам получить только младший бит соответствующего открытого текста. Кажется, что не получится забрать с сервера достаточно полезной информации, так? На самом деле нет. Даже утечки в один бит достаточно, чтобы полностью сломать криптосистему. Вот, как это сделать. Вспомним, что RSA - это гомоморфное шифрование, и значит:
$$\forall M_1,M_2\ \in\ Z^{*}_N,\ C_1C_2\ mod\ N=(M_1M_2)^{e}\ mod\ N=M^{e}_1M^{e}_2\ mod\ N$$
Отдельный случай:
$$ Enc(2M)=2^{e}M^{e}\ mod\ N$$
Мы можем воспользоваться этим свойством для получения открытого текста. Давайте посмотрим, какой младший бит мы получим, если умножим открытый текст $M$ на $2$.
Можно подумать, что он всегда будет $0$, раз младший бит целого числа, умноженного на $2$ - $0$. Но мы умножаем по модулю $N$, и если получившийся открытый текст больше или равен $N$, мы вычитаем из него $N$, пока он не станет меньше, чем $N$. Что конкретно происходит, если мы умножим открытый текст на $2$? Мы знаем, что $M$ изначально меньше, чем $N$:
$$M\lt N$$
Значит $$ 2M\lt 2N$$
и есть два случая: <br><br>
$ 2M\lt N$ и $ 2M \gt N$
<br><br>
$2M \ne N$ так как $N$ - нечётное. <br><br>
В случае когда $2M \lt N$ мы не вычитаем $N$ из результата умножения, и $LSB(2M\ mod\ N)=0$. Однако если $2M \gt N$, то получившийся открытый текст - это $2M - N$, и раз $N$ - нечётное, $LSB(2M\ mod\ N)=1$.
Таким образом мы можем определить, что $2M\lt N$ или что $2M \gt N$, что значит $M\lt \lceil\frac{N}{2}\rceil$ или $M\ge \lceil\frac{N}{2}\rceil$. Мы можем разделить область значений $M$ пополам. И мы можем продолжить этот процесс.
Если $LSB(4M)=0$, то $M$ находится в левой половине ранее поделенной пополам области значений, а если $LSB(4M)=1$, то он находится в правой половине. Мы можем продолжать этот процесс пока либо не узнаем $M$ или область значений $M$ будет настолько маленькой, что легче будет проверить все значения, чем запрашивать биты с сервера.

Вам предоставлен сервер, который представляет собой Оракул по младшему биту. Вы можете отправлять закрытые тексты и он вернёт младший бит открытого текста. Восстановите изначальный открытый текст, используя оракул. Удачи!

In [114]:
import socket
import re
from Crypto.Util.number import inverse, long_to_bytes, bytes_to_long
class VulnServerClient:
    def __init__(self,show=True):
        """Инициализация, подключаемся к серверу"""
        self.s=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
        self.s.connect(('cryptotraining.zone',1343))
        self.s.settimeout(10.0)
        banner = self.recv_until()
        if show:
            print(banner.decode())

    def recv_until(self, symb=b'\n>'):
        """Получаем сообщения с сервера, по дефолту до знака приглашения"""
        data = b''
        while True:
            try:
                chunk = self.s.recv(1)
                if not chunk:
                    raise socket.error("Connection closed")
                data += chunk
                if data[-len(symb):] == symb:
                    break
            except socket.timeout:
                raise
        return data
    
    def reconnect(self):
        """Переподключение к серверу"""
        try:
            self.s.close()
        except:
            pass
        self.s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.s.settimeout(5.0)
        self.s.connect(('cryptotraining.zone', 1343))
        self.recv_until()

    def get_public_key(self,show=True):
        """Получаем открытый ключ с сервера"""
        self.s.sendall('public\n'.encode())
        response=self.recv_until().decode()
        if show:
            print(response)
        e=int(re.search('(?<=e: )\d+',response).group(0))
        N=int(re.search('(?<=N: )\d+',response).group(0))
        return (e,N)
    
    def get_ciphertext(self,show=True):
        """Получаем шифротекст с сервера"""
        self.s.sendall('ciphertext\n'.encode())
        response=self.recv_until().decode()
        if show:
            print (response)
        c=bytes_to_long(bytes.fromhex(re.search('(?<=ciphertext: )[0-9a-f]+',response).group(0)))
        return c
    
    def get_LSB_from_ciphertext(self, ciphertext, show=True):
        """Получаем младший бит открытого текста для соответствующего шифротекста"""
        if isinstance(ciphertext,int):
            ciphertext=long_to_bytes(ciphertext)
        if len(ciphertext)>256:
            print ('Ciphertext too long')
            return None
        if len (ciphertext)<256:
            ciphertext=bytes([0]*(256-len(ciphertext)))+ciphertext
        ciphertext_hex=ciphertext.hex()
        
        self.s.sendall(f'lsb {ciphertext_hex}\n'.encode())
        
        response=self.recv_until().decode()
        if show:
            print (response)
        lsb=int(re.search(r'(?<=lsb is: )\d',response).group(0))

        return lsb
    
    def __del__(self):
        self.s.close()

<>:46: SyntaxWarning: invalid escape sequence '\d'
<>:47: SyntaxWarning: invalid escape sequence '\d'
<>:46: SyntaxWarning: invalid escape sequence '\d'
<>:47: SyntaxWarning: invalid escape sequence '\d'
/var/folders/jm/fq05587x6d787nhsnkqr2zy40000gp/T/ipykernel_30480/1070140320.py:46: SyntaxWarning: invalid escape sequence '\d'
  e=int(re.search('(?<=e: )\d+',response).group(0))
/var/folders/jm/fq05587x6d787nhsnkqr2zy40000gp/T/ipykernel_30480/1070140320.py:47: SyntaxWarning: invalid escape sequence '\d'
  N=int(re.search('(?<=N: )\d+',response).group(0))


In [115]:
vs=VulnServerClient(show=False)
(e,N)=vs.get_public_key(show=False)
c=vs.get_ciphertext(show=False)
print(c)
print (vs.get_LSB_from_ciphertext(c, False))
# Советую использовать опцию show=False. 2000 запросов с полным выводом - слишком много.

25257364269259409757470361992736754958866298375716549729093545690016095602359189147839961332547423807307269559134871591985827523588038074962249500505744121709261571159775363035570492047111938895696096936182016442401259139381881843682997015799674111099482183870160375650995802421684666193992161604681315665462306899393791216364853482708505089200996648387608247245964347122388458890659047550114037323589029924380235750577948907281213048487251119271828247078230102621905412009112071398657067083024386779651707877378636751674301014210033525074488687834082846685330748678104971124483585633186722180674024645384401655433979
0


Поиск

In [ ]:
from Crypto.Util.number import long_to_bytes
import socket
import time

vs = VulnServerClient(show=False)
e, N = vs.get_public_key(show=False)
c = vs.get_ciphertext(show=False)
print(f"Start: N_bits={N.bit_length()}")

mul = pow(2, e, N)
lower, upper = 0, N
c_curr = c

for k in range(1, N.bit_length() + 1):
    try:
        c_curr = (c_curr * mul) % N
        lsb = vs.get_LSB_from_ciphertext(c_curr, show=False)
    except (socket.timeout, ConnectionResetError, ValueError) as ex:
        print(f"\n[!] Error at iter {k}: {ex}")
        print("Reconnecting...")
        vs.reconnect()
        time.sleep(5)
        lsb = vs.get_LSB_from_ciphertext(c_curr, show=False)
    
    mid = (lower + upper) // 2
    if lsb == 0:
        upper = mid
    else:
        lower = mid
    
    if k % 200 == 0:
        print(f"Iter {k}: width = {upper - lower}")
    if k % 100 == 0:
        time.sleep(0.1)


Результат:

In [122]:
print(f"lower: {lower.bit_length()}")
print(f"upper: {upper.bit_length()}")
print(f"range: {upper - lower}")

M = (lower + upper) // 2
print(f"\nDone! M.bit_length() = {M.bit_length()}")
print(f"Flag: {long_to_bytes(M)}")

lower: 1735
upper: 1735
range: 1

Done! M.bit_length() = 1735
Flag: b"Congratulations! Here is your flag: CRYPTOTRAINING{3v3n_4_5m4ll_l34k_c4n_l34d_70_hug3_c0n53q3nc35}. Also some placeholder text, because I need it to be pretty large and I don't want to use paddings. NOT YET at least-H"
